In [65]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

We are using os and sys because by default jupyter work outside the project root so we need to make sure that it can also search the given path.

In [66]:
import os
import sys

sys.path.append(os.path.abspath("../.."))

from config import FEATURED_BTC_DATA_PATH

In [67]:
old_df = pd.read_csv(FEATURED_BTC_DATA_PATH)

In [68]:
old_df.shape

(34912, 56)

In [69]:
old_df.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume', 'quote_volume',
       'number_of_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume',
       'taker_buy_ratio', 'fng_value', 'onchain_num_tx', 'onchain_hash_rate',
       'onchain_miners_revenue_usd', 'Price Change', 'Volatility',
       'high_low_ratio', 'close_open_diff', 'Rolling_Mean_Close',
       'Rolling_Std_Close', 'rolling_max_6h', 'rolling_min_6h', 'Target_Close',
       'Target_Movement', 'Target_Return', 'vol_1h', 'vol_mean_6h',
       'vol_std_6h', 'vol_max_6h', 'vol_min_6h', 'return_1h_lag',
       'return_mean_6h', 'return_std_6h', 'hour', 'dayofweek', 'day',
       'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'rsi', 'macd', 'bb_high',
       'bb_low', 'ema_10', 'ema_30', 'close_lag_1', 'volume_lag_1',
       'price_volatility_interaction', 'taker_buy_ratio_mean_6h',
       'trades_mean_6h', 'fng_mean_3d', 'onchain_num_tx_change',
       'onchain_hash_rate_change', 'onchain_miners_revenue_change'],
  

In [70]:
columns = old_df.select_dtypes(include = [np.number])

Here in heatmap we can see that there is one column is duplicate of other column also we can see column whose value are in dollers are in 0.99 to 1.0.

In [71]:
plt.figure(figsize = (30,30))
sb.heatmap(columns.corr(), annot = True, cmap = "coolwarm", fmt = ".2f")
plt.title("Correlation heatmap to find duplicate columns")
plt.show()

C:\Users\2335m\AppData\Local\Temp\ipykernel_19960\1993386455.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Here is we can see some columns are also giving flat value to train model

In [72]:
num_col = old_df.select_dtypes(include=[np.number]).drop(columns = ["Target_Movement", "Target_Return"], errors = "ignore")
target_corr = num_col.apply(lambda col: col.corr(old_df["Target_Movement"])).sort_values()

In [73]:
plt.figure(figsize=(8, 10))
sb.barplot(x=target_corr.values, y=target_corr.index, palette="vlag")
plt.axvline(0, color="black", linestyle="--")
plt.title("Signal Strength: Feature Correlation with Target_Movement")
plt.xlabel("Correlation (Zero = Not Worth It)")
plt.show()

C:\Users\2335m\AppData\Local\Temp\ipykernel_19960\4007063419.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sb.barplot(x=target_corr.values, y=target_corr.index, palette="vlag")
C:\Users\2335m\AppData\Local\Temp\ipykernel_19960\4007063419.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Now we need to convert doller into percentage

In [74]:
df = old_df.copy()

In [75]:
df["candle_body_pct"] = (df["close"] - df["open"]) / df["open"]
df["hl_range_pct"] = (df["high"] - df["low"]) / df["close"]

In [76]:
df["sma_10_dist_pct"] = (df["close"] - df["Rolling_Mean_Close"]) / df["Rolling_Mean_Close"]
df["ema_10_dist_pct"] = (df["close"] - df["ema_10"]) / df["ema_10"]
df["ema_30_dist_pct"] = (df["close"] - df["ema_30"]) / df["ema_30"]


In [77]:
df["bb_width_pct"] = (df["bb_high"] - df["bb_low"]) / df["close"]
df["bb_pct_b"] = (df["close"] - df["bb_low"]) / (df["bb_high"] - df["bb_low"])

These features need to be dropped after seeing above visual and some other features also as they represent raw-price levels, nominal currency moving averages or dollar-denominated aggregates

In [78]:
dropped = ["open", "high", "low", "close_lag_1",
"Rolling_Mean_Close", "Rolling_Std_Close", "rolling_max_6h",
"rolling_min_6h", "bb_high", "bb_low", "ema_10", "ema_30",
"quote_volume", "taker_buy_quote_volume",
"close_open_diff", "day", "fng_value", "fng_mean_3d", 
"onchain_num_tx_change", "onchain_hash_rate_change", 
"onchain_miners_revenue_change", "onchain_miners_revenue_usd"
]

In [79]:
df = df.drop(columns=dropped)

In [80]:
len(df.columns)

41

Now we are going to use chronological 3 way split 

In [81]:
df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

In [82]:
n = len(df)

In [83]:
train_end = int(n * 0.60)
val_end = train_end + 1 + int(n * 0.20)

In [84]:
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end + 1 : val_end]
holdout_df = df.iloc[val_end + 1:]

Training the model

In [85]:
features_col = [col for col in df.columns if col not in ["timestamp", "Target_Movement", "Target_Return", "Target_Close", "close"]]

In [86]:
from xgboost import XGBClassifier, XGBRegressor

In [87]:
reg = XGBRegressor(
    objective = "reg:pseudohubererror",
    learning_rate = 0.015,
    max_depth = 3,
    min_child_weight = 25,
    reg_lambda = 5.0
)


In [88]:
clf = XGBClassifier(
    eval_metric = "logloss",
    learning_rate = 0.015,
    max_depth = 4,
    min_child_weight = 20,
    reg_lambda = 5.0,
    reg_alpha = 2.0
)

In [89]:
reg.fit(train_df[features_col], train_df["Target_Return"])
clf.fit(train_df[features_col], train_df["Target_Movement"])

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.015, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=20, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [90]:
val_pred_returns = reg.predict(val_df[features_col])
val_pred_prob = clf.predict_proba(val_df[features_col])[:, 1]
val_pred_dir = (val_pred_prob >= 0.50).astype(int)

Validation

In [91]:
from sklearn.metrics import accuracy_score, f1_score, log_loss, mean_squared_error

In [92]:
y_true_dir = val_df["Target_Movement"].astype(int).values
y_true_ret = val_df["Target_Return"].values

In [93]:
acc = accuracy_score(y_true_dir, val_pred_dir) * 100.0
macro_f1 = f1_score(y_true_dir, val_pred_dir, average="macro")
loss = log_loss(y_true_dir, val_pred_prob)

In [94]:
reg_dir_hit = np.mean((val_pred_returns >= 0) == (y_true_ret >= 0)) * 100.0

In [95]:
print(f"Classifier Accuracy: {acc:.2f}%")
print(f"Classifier Macro F1: {macro_f1:.4f}")
print(f"Classifier Log-Loss: {loss:.4f}  (Baseline coin-flip is 0.6931)")
print(f"Regressor Directional Hit: {reg_dir_hit:.2f}%")

Classifier Accuracy: 52.95%
Classifier Macro F1: 0.5240
Classifier Log-Loss: 0.6910  (Baseline coin-flip is 0.6931)
Regressor Directional Hit: 50.79%


Now we are going to introduce fee in this training

In [96]:
MIN_HURDLE = 0.00139951

In [97]:
len_of_train_df = len(train_df)
split_of_train_df = int(len_of_train_df * 0.85)
train_sub_df = train_df.iloc[:split_of_train_df]
calib_df = train_df.iloc[split_of_train_df + 1:]

In [98]:
train_sub_rets = train_sub_df["Target_Return"].values
train_sub_weights = np.clip(np.abs(train_sub_rets) / MIN_HURDLE, 0.25, 3.0).astype(np.float32)

In [99]:
clf_cost_aware = XGBClassifier(
    n_estimators=250,        # Number of trees to build (Default is 100; 250 gives deeper learning)
    learning_rate=0.015,     # Step size per tree
    max_depth=4,             # Tree depth (prevents memorizing noise)
    min_child_weight=20,     # Minimum 20 candles required to make a split
    subsample=0.80,          # Randomly picks 80% of rows per tree (Bagging: prevents overfitting to specific weeks)
    colsample_bytree=0.70,   # Randomly picks 70% of features per tree (Stops one feature from dominating)
    gamma=0.20,              # Minimum gain required to split a node (Tree pruning)
    reg_alpha=2.0,           # L1 Regularization (forces dead features to 0)
    reg_lambda=5.0,          # L2 Regularization (shrinks extreme weights)
    random_state=42,         # Ensures identical, reproducible results every time you run it
    n_jobs=-1,               # Uses all your CPU cores to train in seconds
    eval_metric="logloss"    # Optimizes probability calibration
)

In [100]:
clf_cost_aware.fit(train_sub_df[features_col], 
train_sub_df["Target_Movement"], sample_weight=train_sub_weights)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=0.2, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.015, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=20, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=250,
              n_jobs=-1, num_parallel_tree=None, random_state=42, ...)

In [101]:
val_pred2_prob = clf_cost_aware.predict_proba(val_df[features_col])[:, 1]
val_pred2_dir = (val_pred2_prob >= 0.50).astype(int)

In [102]:
acc = accuracy_score(y_true_dir, val_pred2_dir) * 100.0
macro_f1 = f1_score(y_true_dir, val_pred2_dir, average="macro")
loss = log_loss(y_true_dir, val_pred2_prob)
print(f"Classifier Accuracy: {acc:.2f}%")
print(f"Classifier Macro F1: {macro_f1:.4f}")
print(f"Classifier Log-Loss: {loss:.4f}  (Baseline coin-flip is 0.6931)")

Classifier Accuracy: 52.48%
Classifier Macro F1: 0.4986
Classifier Log-Loss: 0.6945  (Baseline coin-flip is 0.6931)


Now we are going to see how much my model is confident in up and down

In [103]:
q_high = np.quantile(val_pred2_prob, 0.75) 
q_low = np.quantile(val_pred2_prob, 0.25) 

In [104]:
high_conv_mask = (val_pred2_prob >= q_high) | (val_pred2_prob <= q_low)

In [105]:
high_conv_acc = accuracy_score(y_true_dir[high_conv_mask], val_pred2_dir[high_conv_mask]) * 100.0
print(f"Standard Accuracy (All hours):          {acc:.2f}%")
print(f"High-Conviction Accuracy (Top 50% calls): {high_conv_acc:.2f}%")


Standard Accuracy (All hours):          52.48%
High-Conviction Accuracy (Top 50% calls): 54.32%


5-fold Walk forward validation

In [106]:
import src.backtesting
from experiments.backtesting.walk_forward_validate import run_walk_forward_validation

In [107]:
all_weights = np.clip(np.abs(df["Target_Return"].values) / MIN_HURDLE, 0.25, 3.0)

In [108]:
wf_results_df = run_walk_forward_validation(
    df=df,
    model=clf_cost_aware,
    feature_cols=features_col,
    threshold=0.50,
    sample_weight=all_weights
)
display(wf_results_df)

[INFO] Evaluating Window 1/5 (2022-07-09 09:00:00 -> 2023-04-26 13:00:00)...
[INFO] Evaluating Window 2/5 (2023-04-26 14:00:00 -> 2024-02-11 11:00:00)...
[INFO] Evaluating Window 3/5 (2024-02-11 12:00:00 -> 2024-11-28 09:00:00)...
[INFO] Evaluating Window 4/5 (2024-11-28 10:00:00 -> 2025-09-15 07:00:00)...
[INFO] Evaluating Window 5/5 (2025-09-15 08:00:00 -> 2026-07-06 08:00:00)...


,window,period_start,period_end,num_trades,win_rate_pct,total_return_pct,buy_hold_return_pct,sharpe_ratio_annualized
0,1,2022-07-09 09:00:00,2023-04-26 13:00:00,490,42.45,-41.08,79.98,-5.446
1,2,2023-04-26 14:00:00,2024-02-11 11:00:00,482,52.70,-28.61,69.78,-3.559
2,3,2024-02-11 12:00:00,2024-11-28 09:00:00,504,55.75,-7.20,56.36,-0.350
3,4,2024-11-28 10:00:00,2025-09-15 07:00:00,430,40.93,-45.73,5.62,-7.631
4,5,2025-09-15 08:00:00,2026-07-06 08:00:00,519,42.77,-51.72,-10.38,-7.128


Because my model over trade we are going to introduce platt / sigmoid probability calibration

In [109]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, brier_score_loss

In [110]:
calib_rets = calib_df["Target_Return"].values
calib_weights = np.clip(np.abs(calib_rets) / MIN_HURDLE, 0.25, 3.0).astype(np.float32)

In [111]:
calibrated_clf = CalibratedClassifierCV(
    estimator=clf_cost_aware,
    method="sigmoid",
    cv="prefit"
)

In [112]:
calibrated_clf.fit(calib_df[features_col], calib_df["Target_Movement"], sample_weight = calib_weights.astype(np.float32))

CalibratedClassifierCV(cv='prefit',
                       estimator=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=0.7,
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=False,
                                               eval_metric='logloss',
                                               feature_types=None, gamma=0.2,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=0.015,
                                               max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None, max_depth=4,
                                               max_leaves=None,
                                               min_child_weight=20, missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=250, n_jobs=-1,
                                               num_parallel_tree=None,
                                               random_state=42, ...))

In [113]:
raw_probs = clf_cost_aware.predict_proba(val_df[features_col])[:, 1]
cal_probs = calibrated_clf.predict_proba(val_df[features_col])[:, 1]
print(f"Raw Log-Loss:           {log_loss(y_true_dir, raw_probs):.4f}")
print(f"Calibrated Log-Loss:    {log_loss(y_true_dir, cal_probs):.4f} (Closer to 0 is better)")
print(f"Raw Brier Score:        {brier_score_loss(y_true_dir, raw_probs):.4f}")
print(f"Calibrated Brier Score: {brier_score_loss(y_true_dir, cal_probs):.4f} (Lower is better)")

Raw Log-Loss:           0.6945
Calibrated Log-Loss:    0.6926 (Closer to 0 is better)
Raw Brier Score:        0.2506
Calibrated Brier Score: 0.2497 (Lower is better)


Again we are going to back test after calibrating

In [114]:
from src.backtesting import run_backtest, compute_metrics

In [115]:
threshold_results = []
for thresh in [0.50, 0.51, 0.52, 0.53]:
    ec, tl = run_backtest(
        val_df, 
        calibrated_clf, 
        threshold=thresh, 
        feature_cols=features_col
    )
    m = compute_metrics(ec, tl, val_df)
    threshold_results.append({
        "Threshold": thresh,
        "Trades": m["num_trades"],
        "Win Rate (%)": round(m["win_rate_pct"], 2),
        "Total Return (%)": round(m["total_return_pct"], 2),
        "Buy & Hold (%)": round(m["bh_return_pct"], 2),
        "Max Drawdown (%)": round(m["max_drawdown_pct"], 2),
        "Sharpe": round(m["sharpe_annualized"], 2)
    })

In [116]:
threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

,Threshold,Trades,Win Rate (%),Total Return (%),Buy & Hold (%),Max Drawdown (%),Sharpe
0,0.50,1042,51.82,-57.77,20.3,-58.29,-2.80
1,0.51,1145,51.09,-59.54,20.3,-59.85,-3.17
2,0.52,1204,48.17,-70.21,20.3,-70.71,-4.60
3,0.53,1253,46.53,-72.63,20.3,-72.90,-5.19


Now we are going to use significant test file

In [117]:
ec_50, tl_50 = run_backtest(
    val_df, 
    calibrated_clf, 
    threshold=0.50, 
    feature_cols=features_col
)

In [118]:
from experiments.backtesting.significance_test import evaluate_trade_significance

In [119]:
sig_report = evaluate_trade_significance(tl_50, alpha=0.05)


[STATISTIC] Total Trades: 1042 | Wins: 540 | Win Rate: 51.8%
[STATISTIC] Binomial Test vs p=0.5 (H1: win rate > 50%): p-value = 0.1258
  -> NOT statistically significant (indistinguishable from random noise)
[STATISTIC] Gross Per-Trade Return: mean=0.061%, 95% Bootstrap CI=[0.006%, 0.117%]
  -> CI strictly excludes 0 (empirical gross edge detected)
[STATISTIC] Net Per-Trade Return (Post-Friction): mean=-0.079%, 95% Bootstrap CI=[-0.134%, -0.023%]
  -> Net CI spans 0 (edge fully consumed by transaction friction)


In [120]:
prob_series = pd.Series(cal_probs)
print(prob_series.describe(percentiles=[0.50, 0.75, 0.85, 0.90, 0.95, 0.99]))

count    6982.000000
mean        0.524938
std         0.053399
min         0.341691
50%         0.530638
75%         0.563253
85%         0.579223
90%         0.589552
95%         0.604584
99%         0.628959
max         0.669208
dtype: float64


In [121]:
high_conv_results = []
for thresh in [0.54, 0.55, 0.56, 0.57, 0.58]:
    ec, tl = run_backtest(
        val_df, 
        calibrated_clf, 
        threshold=thresh, 
        feature_cols=features_col
    )
    m = compute_metrics(ec, tl, val_df)
    high_conv_results.append({
        "Threshold": thresh,
        "Trades": m["num_trades"],
        "Win Rate (%)": round(m["win_rate_pct"], 2),
        "Total Return (%)": round(m["total_return_pct"], 2),
        "Buy & Hold (%)": round(m["bh_return_pct"], 2),
        "Max Drawdown (%)": round(m["max_drawdown_pct"], 2),
        "Sharpe": round(m["sharpe_annualized"], 2)
    })
high_conv_df = pd.DataFrame(high_conv_results)
display(high_conv_df)

,Threshold,Trades,Win Rate (%),Total Return (%),Buy & Hold (%),Max Drawdown (%),Sharpe
0,0.54,1242,43.64,-73.11,20.3,-73.23,-5.71
1,0.55,1166,42.02,-71.42,20.3,-71.50,-6.05
2,0.56,1014,42.11,-66.98,20.3,-67.08,-6.06
3,0.57,839,40.88,-64.17,20.3,-64.28,-6.11
4,0.58,646,44.12,-38.72,20.3,-39.76,-3.46


Now we are going to use and test hysteresis across different entry threshold

In [122]:
from src.backtesting import _simulate_from_signals

In [123]:
hysteresis_results = []
for entry_thresh in [0.52, 0.53, 0.54, 0.55]:
    signals = np.zeros(len(val_df), dtype=int)
    pos = 0
    for i, p in enumerate(cal_probs):
        if pos == 0 and p >= entry_thresh:
            pos = 1  # Enter on high conviction
        elif pos == 1 and p < 0.50:
            pos = 0  # Exit only when trend flips bearish
        signals[i] = pos
        
    ec_h, tl_h = _simulate_from_signals(val_df, signals)
    m_h = compute_metrics(ec_h, tl_h, val_df)
    
    hysteresis_results.append({
        "Entry Threshold": entry_thresh,
        "Exit Threshold": 0.50,
        "Trades": m_h["num_trades"],
        "Win Rate (%)": round(m_h["win_rate_pct"], 2),
        "Total Return (%)": round(m_h["total_return_pct"], 2),
        "Buy & Hold (%)": round(m_h["bh_return_pct"], 2),
        "Max Drawdown (%)": round(m_h["max_drawdown_pct"], 2),
        "Sharpe": round(m_h["sharpe_annualized"], 2)
    })

In [124]:
hyst_df = pd.DataFrame(hysteresis_results)
display(hyst_df)

,Entry Threshold,Exit Threshold,Trades,Win Rate (%),Total Return (%),Buy & Hold (%),Max Drawdown (%),Sharpe
0,0.52,0.5,912,53.73,-45.55,20.3,-46.91,-2.03
1,0.53,0.5,844,54.74,-37.29,20.3,-40.36,-1.55
2,0.54,0.5,765,56.08,-34.53,20.3,-43.47,-1.46
3,0.55,0.5,684,57.60,-21.13,20.3,-32.08,-0.80


Now we are going test if higher entry conviction and slightly wider exit will drop the fees cost

In [125]:
fine_hyst_results = []
for entry_thresh in [0.55, 0.56, 0.57, 0.58]:
    for exit_thresh in [0.50, 0.48]:
        signals = np.zeros(len(val_df), dtype=int)
        pos = 0
        for i, p in enumerate(cal_probs):
            if pos == 0 and p >= entry_thresh:
                pos = 1  # Enter only on top-quantile conviction
            elif pos == 1 and p < exit_thresh:
                pos = 0  # Exit only when trend flips firmly bearish
            signals[i] = pos
            
        ec_h, tl_h = _simulate_from_signals(val_df, signals)
        m_h = compute_metrics(ec_h, tl_h, val_df)
        
        fine_hyst_results.append({
            "Entry": entry_thresh,
            "Exit": exit_thresh,
            "Trades": m_h["num_trades"],
            "Win Rate (%)": round(m_h["win_rate_pct"], 2),
            "Total Return (%)": round(m_h["total_return_pct"], 2),
            "Buy & Hold (%)": round(m_h["bh_return_pct"], 2),
            "Max Drawdown (%)": round(m_h["max_drawdown_pct"], 2),
            "Sharpe": round(m_h["sharpe_annualized"], 2)
        })
fine_hyst_df = pd.DataFrame(fine_hyst_results)
display(fine_hyst_df)

,Entry,Exit,Trades,Win Rate (%),Total Return (%),Buy & Hold (%),Max Drawdown (%),Sharpe
0,0.55,0.50,684,57.60,-21.13,20.3,-32.08,-0.80
1,0.55,0.48,527,62.24,-10.12,20.3,-28.66,-0.23
2,0.56,0.50,586,58.70,-7.75,20.3,-23.69,-0.20
3,0.56,0.48,456,61.62,-7.20,20.3,-24.58,-0.13
4,0.57,0.50,492,61.18,-4.95,20.3,-27.31,-0.09
5,0.57,0.48,391,62.92,2.34,20.3,-21.78,0.25
6,0.58,0.50,403,62.53,25.33,20.3,-18.66,1.26
7,0.58,0.48,330,64.85,28.76,20.3,-15.36,1.26


Now we're going to focus in regression are introducing Uh pasudo harbor error and also by introducing fees

In [126]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [127]:
reg_cost_aware = XGBRegressor(
    objective = "reg:pseudohubererror",
    n_estimators = 250,
    learning_rate = 0.015,
    max_depth = 4,
    subsample = 0.80,
    colsample_bytree = 0.70,
    random_state = 42,
    n_jobs = 1
)

In [128]:
reg_cost_aware.fit(train_sub_df[features_col], train_sub_df["Target_Return"], sample_weight=train_sub_weights)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.015, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=250, n_jobs=1,
             num_parallel_tree=None, objective='reg:pseudohubererror', ...)

In [129]:
val_pred2_ret = reg_cost_aware.predict(val_df[features_col])

In [130]:
MAE = mean_absolute_error(val_df["Target_Return"], val_pred2_ret)
dir_hit_rate = (np.sign(val_pred2_ret) == np.sign(val_df["Target_Return"])).mean() * 100

In [131]:
print(MAE)
print(dir_hit_rate)

0.0032195424255499843
49.39845316528215


Now we are going to test our regression model bit we need to change its signal to 0 and 1 for that we are going to change the value by math formula where if predicted return price is smaller the fee hurdle signal will be 0 else 1.

In [132]:
val_ret_sig = (val_pred2_ret >= 0.0014).astype(int)

In [133]:
ec_reg, tl_reg = _simulate_from_signals(val_df, val_ret_sig)
mh_reg = compute_metrics(ec_reg, tl_reg, val_df)

In [134]:
print(mh_reg)

{'initial_capital': 10000.0, 'final_equity': 8813.16, 'total_return_pct': -11.87, 'bh_final_equity': 12030.14, 'bh_return_pct': 20.3, 'sharpe_annualized': -0.912, 'sortino_annualized': -0.323, 'max_drawdown_pct': -19.69, 'bh_max_drawdown_pct': -30.94, 'calmar_ratio': -0.603, 'win_rate_pct': 44.03, 'profit_factor': 0.78, 'num_trades': 134, 'avg_trade_return_pct': -0.088, 'exposure_pct': 3.95}


Now we are going to find which features are important for our model

In [135]:
clf_dic = {}
reg_dic = {}
imp_clf_fea = clf_cost_aware.feature_importances_
imp_reg_fea = reg_cost_aware.feature_importances_
clf_dic = dict(zip(features_col, imp_clf_fea))
reg_dic = dict(zip(features_col, imp_reg_fea))
print(sorted(clf_dic.items(), key = lambda x : x[1])[-10:])
print(sorted(reg_dic.items(), key = lambda x : x[1])[-10:])

[('return_std_6h', 0.028245835), ('dayofweek', 0.028366841), ('vol_std_6h', 0.028891413), ('taker_buy_base_volume', 0.029862907), ('return_mean_6h', 0.030119995), ('Price Change', 0.032645337), ('taker_buy_ratio', 0.036178954), ('return_1h_lag', 0.036378972), ('candle_body_pct', 0.04363254), ('sma_10_dist_pct', 0.054997824)]
[('taker_buy_ratio_mean_6h', 0.029470725), ('vol_1h', 0.029989872), ('volume_lag_1', 0.030073073), ('hour_cos', 0.030828478), ('Volatility', 0.031697944), ('ema_30_dist_pct', 0.03209311), ('high_low_ratio', 0.032558728), ('vol_std_6h', 0.033279978), ('vol_max_6h', 0.035639446), ('hour_sin', 0.037859235)]


After understanding which features are important and helping our model to understand also as we can see that every feature is important so we are not going to remove anything.

Now we are going to combine our these two different model to make duel-head model

In [136]:
fused_signals = ((cal_probs >= 0.56) & (val_pred2_ret >= 0.0014)).astype(int)

In [137]:
new_ec, new_tl = _simulate_from_signals(val_df, fused_signals)
new_mh = compute_metrics(new_ec, new_tl, val_df)
print(new_mh)

{'initial_capital': 10000.0, 'final_equity': 9119.0, 'total_return_pct': -8.81, 'bh_final_equity': 12030.14, 'bh_return_pct': 20.3, 'sharpe_annualized': -1.01, 'sortino_annualized': -0.22, 'max_drawdown_pct': -15.21, 'bh_max_drawdown_pct': -30.94, 'calmar_ratio': -0.579, 'win_rate_pct': 52.27, 'profit_factor': 0.73, 'num_trades': 88, 'avg_trade_return_pct': -0.1, 'exposure_pct': 1.73}


Now we're going to do hysteresis

In [138]:
hys_sig = np.zeros(len(val_df), dtype=int)
pos = 0
for pred, row in enumerate(val_df.itertuples()):
    if pos == 0:
        if cal_probs[pred] >= 0.56 and val_pred2_ret[pred] >= 0.0014:
            pos = 1
    elif pos == 1:
        if cal_probs[pred] < 0.48 and val_pred2_ret[pred] < 0.0:
            pos = 0
    hys_sig[pred] = pos
print(hys_sig)

[0 0 0 ... 0 0 0]


In [139]:
new2_ec, new2_tl = _simulate_from_signals(val_df, hys_sig)
new2_mh = compute_metrics(new2_ec, new2_tl, val_df)
print(new2_mh)

{'initial_capital': 10000.0, 'final_equity': 11366.75, 'total_return_pct': 13.67, 'bh_final_equity': 12030.14, 'bh_return_pct': 20.3, 'sharpe_annualized': 0.884, 'sortino_annualized': 0.391, 'max_drawdown_pct': -11.14, 'bh_max_drawdown_pct': -30.94, 'calmar_ratio': 1.227, 'win_rate_pct': 59.65, 'profit_factor': 1.3, 'num_trades': 57, 'avg_trade_return_pct': 0.251, 'exposure_pct': 11.7}


Now we're going to check the binomial P value and bootstrap confidence interval

In [140]:
from experiments.backtesting.significance_test import evaluate_trade_significance

In [141]:
sig_result = evaluate_trade_significance(new2_tl)
print(sig_result)

[STATISTIC] Total Trades: 57 | Wins: 34 | Win Rate: 59.6%
[STATISTIC] Binomial Test vs p=0.5 (H1: win rate > 50%): p-value = 0.0924
  -> NOT statistically significant (indistinguishable from random noise)
[STATISTIC] Gross Per-Trade Return: mean=0.392%, 95% Bootstrap CI=[-0.197%, 0.978%]
  -> CI spans 0 (insufficient evidence of gross edge)
[STATISTIC] Net Per-Trade Return (Post-Friction): mean=0.251%, 95% Bootstrap CI=[-0.336%, 0.837%]
  -> Net CI spans 0 (edge fully consumed by transaction friction)
{'total_trades': 57, 'wins': 34, 'win_rate_pct': 59.65, 'binomial_pvalue': 0.09243, 'is_significant': False, 'gross_mean_pct': 0.392, 'gross_ci_95': (-0.197, 0.978), 'net_mean_pct': 0.251, 'net_ci_95': (-0.336, 0.837)}


Now We are going test out model against the untouched data

In [142]:
holdout_probs = calibrated_clf.predict_proba(holdout_df[features_col])[:, 1]
holdout_pred_ret = reg_cost_aware.predict(holdout_df[features_col])

In [143]:
holdout_sig = np.zeros(len(holdout_df), dtype=int)
pos = 0
for pred, row in enumerate(holdout_df.itertuples()):
    if pos == 0:
        if holdout_probs[pred] >= 0.56 and holdout_pred_ret[pred] >= 0.0014:
            pos = 1
    elif pos == 1:
        if holdout_probs[pred] < 0.48 and holdout_pred_ret[pred] < 0.0:
            pos = 0
    holdout_sig[pred] = pos

hold_ec, hold_tl = _simulate_from_signals(holdout_df, holdout_sig)
hold_mh = compute_metrics(hold_ec, hold_tl, holdout_df)
print(hold_mh)

{'initial_capital': 10000.0, 'final_equity': 8060.12, 'total_return_pct': -19.4, 'bh_final_equity': 5466.59, 'bh_return_pct': -45.33, 'sharpe_annualized': -1.438, 'sortino_annualized': -0.474, 'max_drawdown_pct': -26.52, 'bh_max_drawdown_pct': -53.74, 'calmar_ratio': -0.732, 'win_rate_pct': 51.43, 'profit_factor': 0.5, 'num_trades': 35, 'avg_trade_return_pct': -0.579, 'exposure_pct': 8.09}


In [144]:
hold_sig_result = evaluate_trade_significance(hold_tl)
print(hold_sig_result)

[STATISTIC] Total Trades: 35 | Wins: 18 | Win Rate: 51.4%
[STATISTIC] Binomial Test vs p=0.5 (H1: win rate > 50%): p-value = 0.5000
  -> NOT statistically significant (indistinguishable from random noise)
[STATISTIC] Gross Per-Trade Return: mean=-0.440%, 95% Bootstrap CI=[-1.358%, 0.358%]
  -> CI spans 0 (insufficient evidence of gross edge)
[STATISTIC] Net Per-Trade Return (Post-Friction): mean=-0.579%, 95% Bootstrap CI=[-1.496%, 0.218%]
  -> Net CI spans 0 (edge fully consumed by transaction friction)
{'total_trades': 35, 'wins': 18, 'win_rate_pct': 51.43, 'binomial_pvalue': 0.5, 'is_significant': False, 'gross_mean_pct': -0.44, 'gross_ci_95': (-1.358, 0.358), 'net_mean_pct': -0.579, 'net_ci_95': (-1.496, 0.218)}
